# B2-024 — GPU Workflow and Scientific-ML Capstone

*30 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

This double-length capstone closes Book 2.
It teaches the workflow of Round 2's open-ended model-building problems, accelerator-aware but always reproducible on a CPU, and three task families built from original, seeded generators: semi-supervised image-shape classification, a scientific inverse problem, and mixture-function parameter regression.

## Learning goals

- Write device-agnostic PyTorch, read device-mismatch errors, choose an autocast dtype, estimate training memory from parameter and activation counts, triage out-of-memory failures, checkpoint with optimizer and RNG state, and plan a Colab L4 session.
- Design experiments as falsifiable hypotheses with baselines, one-factor ablations, fixed step budgets, repeated seeds, held-out discipline, and a log.
- Evaluate models with per-example metrics, percentile bootstrap intervals, paired comparisons, ablation tables, robustness checks, and a written evaluation section.
- Use unlabelled data by confidence-thresholded (noisy-student) self-training and by cluster-then-label, and diagnose confirmation bias and imbalance.
- Solve an inverse problem with a forward operator, grid Tikhonov regularization, and a learned inverse checked by physics residuals and coverage.
- Regress mixture parameters with canonicalization, permutation-invariant metrics, a least-squares baseline, an amortized regressor, and predict-then-refine.

## Owned concepts

| Concept | First taught |
|---|---|
| `gpu-colab-l4-workflow` | Session 1 |
| `open-ended-experiment-design` | Session 2 |
| `open-ended-model-evaluation` | Session 3 |
| `semi-supervised-pseudo-labeling` | Session 4 |
| `scientific-ml-inverse-problems` | Session 5 |
| `mixture-parameter-regression` | Session 6 |

## Study route

0. [Book 1 bridge diagnostic](./lessons/00-book1-bridge.ipynb) — 30 minutes.
1. [The GPU workflow on Colab L4](./lessons/01-gpu-workflow-on-colab-l4.ipynb) — 90 minutes. Then p01, p02, p06, p07.
2. [Experiment design under a budget](./lessons/02-experiment-design-under-budget.ipynb) — 90 minutes. Then p03, p12, p16, p23.
3. [Evaluating open-ended models](./lessons/03-evaluating-open-ended-models.ipynb) — 90 minutes. Then p08, p15, p21, p22, p24.
4. [Semi-supervised pseudo-labeling](./lessons/04-semi-supervised-pseudo-labeling.ipynb) — 90 minutes. Then p04, p09, p17, p18, p25.
5. [Scientific inverse problems](./lessons/05-scientific-inverse-problems.ipynb) — 90 minutes. Then p10, p13, p19, p28.
6. [Mixture-function parameter regression](./lessons/06-mixture-parameter-regression.ipynb) — 90 minutes. Then p05, p11, p14, p20, p26, p27.
7. [Review and self-quiz](./review.ipynb) — 60 minutes.

Practice sets: Set A (p01–p05) is retrieval multiple choice; Set B (p06–p15) is constrained coding and proof; Set C (p16–p28) is integrative training, scenario planning, and open-ended challenges.

## Tools taught inside this unit

Book 1 stops short of several tools this unit needs, so the lessons teach them before any practice uses them: device placement and `torch.autocast` (with `bfloat16`, `float16` plus a gradient scaler, and the `amp_dtype=None` CI path); training-memory arithmetic; checkpoints with optimizer and RNG state; the percentile bootstrap and paired comparisons; confidence-thresholded, class-balanced pseudo-label selection and the noisy student; cluster-then-label; the grid operator and Tikhonov's closed form; physics residuals; canonical ordering and permutation-invariant metrics; and least-squares fitting by Adam on log-widths.

## Data, evaluation, and reproducibility contract

The sole seed is `20261022`.
`scripts/generate_capstone_data.py` produces four datasets and two literal results tables into `data/capstone_datasets.json`; `data/capstone_data.py` loads them, verifies their digests, and exposes immutable split ID tuples, per-row SHA-256 hashes, and `split_of(task, row)`.
Its `--check` mode regenerates everything and compares; no trained weights or metrics are stored.

| task | inputs | targets | train / val / other / test |
|---|---|---|---|
| `shapes_supervised` | float32 `(N,1,8,8)` | int64 class | 600 / 150 / — / 150 |
| `shapes_ssl` | float32 `(N,1,8,8)` | int64 class | 30 / 30 / 600 unlabelled / 300 |
| `inverse` | float32 `(N,8)` magnitudes | float32 `(x, y, strength)` | 800 / 200 / — / 200 |
| `mixture` | float32 `(N,32)` values | float32 canonical 9 parameters | 800 / 200 / — / 200 |

Every model-building practice (p16–p22, p26, p27) follows one protocol: fit on training rows, select on validation rows, and reach the locked test rows only through `final_test_score(predict_fn, n_boot=1000, alpha=0.05, task=...)`, exactly once, after all fitting; it returns the metric with a seeded percentile-bootstrap interval.
Baselines are committed code: `baseline_score(task)` gives the labelled-only CNN ($0.8100$ accuracy), grid Tikhonov ($0.2137$ position error), and fixed-initialization least squares ($0.0091$ parameter error) at the frozen seed.
`StepBudget(max_steps)` counts optimizer steps for budgeted work.

## Compute contract

Every notebook runs on a CPU.
Practices marked `compute.policy: optional-colab-l4` (p06, p07, p16, p17, p19, p20) have a CPU correctness path, which CI executes, and an optional `Accelerator extension` section that gives a larger configuration as a replacement config dict for the same code path; CI never runs it.
No lesson or practice downloads data, pretrained weights, or packages; Session 1's forward-only guide to a real latent-diffusion pipeline is markdown only and not assessed.

## How to study

Run lessons in order, answer both checkpoints in every numbered section before reading its collected answers, and attempt each practice only after the session listed in the study route.
Proofs require reasoning in full sentences; coding statements pin function names, shapes, dtypes, seeds, budgets, and tolerances, and every coding response ends with an answer-check section of executable assertions.

## Boundaries

Original problems only: the task families follow the shape of Round 2's open-ended section as described in the reference analysis, but no past-paper text, data, or close paraphrase is used.
No external data, torchvision datasets, model hubs, pretrained weights, or downloads; no `gpu-required` work.